# Optimización por Enjambre de Partículas (PSO) y Animación con Manim

En este notebook aplicaremos Optimización por Enjambre de Partículas (PSO) las funciones Rosenbrock (2d) y Schwefel (3d).

## Funciones Objetivo

### A. Función de Rosenbrock (Generalizada)
La función de Rosenbrock se expresa como:
$$f(\mathbf{x}) = \sum _{i=1}^{N-1}\left( 100\left( x_{i} -x_{i-1}^{2}\right)^{2} +( 1-x_{i-1})^{2}\right)$$
Utilizaremos su versión en **2 dimensiones ($d=2$)** donde $x_0 = x$ y $x_1 = y$

### B. Función de Schwefel
La función de Schwefel se expresa como:
$$f(\mathbf{x}) = 418.9829d-\sum _{i=1}^{d} x_{i}\sin{\sqrt{|x_{i}|}}$$
utilizaremos la version en **3 dimensiones ($d=3$)**.

In [2]:
%pip install manim -q

import numpy as np
from manim import *

# Poner video en notebook
config.media_embed = True


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\user\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [3]:
def rosenbrock_2d(X):
    """
    Calcula la función de Rosenbrock para 2 dimensiones.
    
    Args:
        X (np.ndarray): Matriz de posiciones de tamaño (N_particulas, 2), 
                        donde la columna 0 es 'x' y la columna 1 es 'y'.
                        
    Returns:
        np.ndarray: Arreglo 1D con los costos calculados para cada partícula.
    """
    x, y = X[:, 0], X[:, 1]
    return 100 * (y - x**2)**2 + (1 - x)**2

def rosenbrock_bounds_2d():
    """
    Devuelve los límites del espacio de búsqueda para Rosenbrock 2D.
    
    Returns:
        np.ndarray: Matriz de límites [[x_min, x_max], [y_min, y_max]].
    """
    return np.array([[-2.0, 2.0], [-1.0, 3.0]])

def schwefel_3d(X):
    """
    Calcula la función de Schwefel para 3 dimensiones.
    
    Args:
        X (np.ndarray): Matriz de posiciones de tamaño (N_particulas, 3).
        
    Returns:
        np.ndarray: Arreglo 1D con los costos calculados para cada partícula.
    """
    dim = X.shape[1] 
    sum_term = np.sum(X * np.sin(np.sqrt(np.abs(X))), axis=1)
    return 418.9829 * dim - sum_term

def schwefel_bounds_3d():
    """
    Devuelve los límites del espacio de búsqueda para Schwefel 3D.
    
    Returns:
        np.ndarray: Matriz de límites para x, y, z.
    """
    return np.array([[-500.0, 500.0], [-500.0, 500.0], [-500.0, 500.0]])

## Algoritmo

La clase `PSOManager` implementa el algoritmo siguiendo la siguiente estructura:

### A. Matrices y Dimensiones
Trabajamos en un espacio definido por:
* $N$: Número de partículas.
* $D$: Dimensión de las partículas.

Las variables principales se representan como matrices en $\mathbb{R}^{N \times D}$:
* **$X \in \mathbb{R}^{N \times D}$**: Matriz de posiciones. Cada fila $x^i$ es la posición de una partícula. Inicializacion aleatoria.
* **$V \in \mathbb{R}^{N \times D}$**: Matriz de velocidades. Inicializacion aleatoria.
* **$P \in \mathbb{R}^{N \times D}$**: Matriz con el "mejor valor histórico de cada partícula" (Personal Best). Inicia como $P = X$.
* **$\hat{p}$**: El mejor valor global (Global Best), Inicia como: `p_hat = argmin(f(X))`.
* **$\hat{P} \in \mathbb{R}^{N \times D}$**: Matriz donde el vector óptimo global $\hat{p}$ se copia en todas las $N$ filas para poder calcular los aportes globales a cada particula.

### B. Bucle
En cada iteración, se aplican las siguientes operaciones a todo el enjambre:

1.  **Actualizar Velocidad ($V$) y Posición ($X$):**
    Utilizando las matrices diagonales de números aleatorios $r, \hat{r} \in \mathbb{R}^{N \times D}$, la velocidad se calcula como:
    $$V = wV + c_1 \cdot r \cdot (P - X) + c_2 \cdot \hat{r} \cdot (\hat{P} - X)$$
    donde $w, c_1, c_2$ representan la inercia, el coeficiente cognitivo y el social respectivamente.
    
    Luego, se actualiza la posición: 
    $$X = X + V$$

2. **Actualizar Mejores Personales ($P$):**
    Se evalúa la función objetivo $f$. Para cada partícula $i \in \{1 \dots N\}$:
    `if f(X[i,:]) < f(P[i,:]): P[i,:] = X[i,:]`

3.  **Actualizar Mejor Global ($\hat{p}$):**
    Se revisa si alguna partícula superó el mejor valor global de todo el enjambre:
    `if f(P[i,:]) < f(\hat{p}): \hat{p} = P[i,:]`

4.  **Historial (`X_hist`):**
    Se registra la poblacion en una historia

In [7]:
class PSOManager:
    """
    Administrador del algoritmo de Optimización por Enjambre de Partículas (PSO).
    Mantiene el estado del enjambre y guarda el historial de posiciones para animaciones.
    """
    def __init__(self, obj_func, bounds, num_particles=40, w=0.5, c1=1.5, c2=1.5):
        """
        Inicializa el enjambre de partículas.
        
        Args:
            obj_func (callable): Función objetivo a minimizar.
            bounds (np.ndarray): Matriz de límites del espacio de búsqueda, shape = (dim, 2).
            num_particles (int): Cantidad de partículas en el enjambre.
            w (float): Peso de inercia (mantiene el impulso de la velocidad previa).
            c1 (float): Coeficiente cognitivo (atracción hacia el mejor personal).
            c2 (float): Coeficiente social (atracción hacia el mejor global).
        """
        self.obj_func = obj_func
        self.bounds = bounds
        self.dim = bounds.shape[0]
        self.N = num_particles
        self.w, self.c1, self.c2 = w, c1, c2
        
        # Matrices (Posiciones y Velocidades)
        self.X = np.zeros((self.N, self.dim))
        self.V = np.zeros((self.N, self.dim))
        
        # Inicializar partículas y velocidades componente por componente
        for d in range(self.dim): 
            self.X[:, d] = np.random.uniform(self.bounds[d, 0], self.bounds[d, 1], self.N)
            v_limit = (self.bounds[d, 1] - self.bounds[d, 0]) * 0.1 # Limite de velocidad para no salirse de las cotas
            self.V[:, d] = np.random.uniform(-v_limit, v_limit, self.N)
            
        # Mejor posición personal histórica
        self.P = np.copy(self.X)
        initial_costs = self.obj_func(self.X)
        
        # Mejor posición global histórica
        best_initial_idx = np.argmin(initial_costs)
        self.g = np.copy(self.X[best_initial_idx])
        self.g_cost = initial_costs[best_initial_idx]
        
        # Historial
        self.history_X = [np.copy(self.X)]
        
    def step(self):
        """
        Ejecuta una sola iteración del algoritmo: actualiza velocidades, 
        posiciones y memorias (P y g).
        """
        # Vectores aleatorios r1 y r2 entre [0, 1]
        r1 = np.random.rand(self.N, self.dim)
        r2 = np.random.rand(self.N, self.dim)
        
        # Update de Velocidad: V = w*V + c1*r1*(P - X) + c2*r2*(g - X)
        self.V = self.w * self.V + self.c1 * r1 * (self.P - self.X) + self.c2 * r2 * (self.g - self.X)
        
        # Limitar la velocidad máxima (Despues pueden "Explotar")
        v_max = (self.bounds[:, 1] - self.bounds[:, 0]) * 0.5
        for d in range(self.dim):
            self.V[:, d] = np.clip(self.V[:, d], -v_max[d], v_max[d])
            
        # Update de Posición: X = X + V
        self.X = self.X + self.V
        
        # Corregimos particulas que se salgan de las cotas
        for d in range(self.dim):
            self.X[:, d] = np.clip(self.X[:, d], self.bounds[d, 0], self.bounds[d, 1])
            
        # Nuevos costos
        current_costs = self.obj_func(self.X)
        personal_best_costs = self.obj_func(self.P)
        
        # Update mejor personal (P)
        improved_mask = current_costs < personal_best_costs
        self.P[improved_mask] = self.X[improved_mask]
        
        # Update mejor global (g)
        new_pb_costs = self.obj_func(self.P) # Otra vez, por el update de arriba
        best_idx = np.argmin(new_pb_costs)
        
        if new_pb_costs[best_idx] < self.g_cost:
            self.g = np.copy(self.P[best_idx])
            self.g_cost = new_pb_costs[best_idx]
            
        # Guardar para historial
        self.history_X.append(np.copy(self.X))

    def run(self, max_iter=100):
        """
        Ejecuta el ciclo de optimización por un número definido de iteraciones.
        
        Args:
            max_iter (int): Número de iteraciones a ejecutar.
            
        Returns:
            tuple: (Mejor posición global encontrada, Costo de esa posición).
        """
        for _ in range(max_iter):
            self.step()
        return self.g, self.g_cost

In [11]:
# Rosenbrock
print("--- Resultados Rosenbrock (2D) ---")
pso_rb = PSOManager(rosenbrock_2d, rosenbrock_bounds_2d(), num_particles=30, w=0.7, c1=1.5, c2=1.5)
mejor_pos_rb, mejor_costo_rb = pso_rb.run(max_iter=200)
print(f"Óptimo encontrado en (x, y): {mejor_pos_rb}")
print(f"Valor de la función: {mejor_costo_rb:.6f}\n")

# Schwefel
print("--- Resultados Schwefel (3D) ---")
pso_sc = PSOManager(schwefel_3d, schwefel_bounds_3d(), num_particles=60, w=0.7, c1=1.5, c2=1.5)
mejor_pos_sc, mejor_costo_sc = pso_sc.run(max_iter=200)
print(f"Óptimo encontrado en (x, y, z): {mejor_pos_sc}")
print(f"Valor de la función: {mejor_costo_sc:.6f}")

--- Resultados Rosenbrock (2D) ---
Óptimo encontrado en (x, y): [1.00003483 1.00006829]
Valor de la función: 0.000000

--- Resultados Schwefel (3D) ---
Óptimo encontrado en (x, y, z): [ 420.9687462   420.96874644 -302.52493534]
Valor de la función: 118.438373


## Animación con Manim
